# Dataset Inspection

Phase 2 read-only inspection of the UCI Appliances Energy Prediction CSV. The raw file is loaded without writing or cleaning it; datetime parsing is kept in a separate in-memory Series.

## Variable groups and unit caveats

| Group | Variables |
| --- | --- |
| Target | `Appliances` |
| Indoor temperature | `T1`-`T5`, `T7`-`T9` |
| Indoor relative humidity | `RH_1`-`RH_5`, `RH_7`-`RH_9` |
| Outdoor/weather | `T6`, `RH_6`, `T_out`, `Press_mm_hg`, `RH_out`, `Windspeed`, `Visibility`, `Tdewpoint` |
| Lighting | `lights` |
| Temporal | `date` |
| Intentionally random/questionable | `rv1`, `rv2` |

Grouping follows conventional UCI dataset descriptions. Detailed sensor locations and most units are not confirmed by metadata in this repository. `Press_mm_hg` indicates mm Hg in its column name; units for the other measurements require authoritative source verification. The local file has `rv1` and `rv2` identical row by row; neither is removed or assumed useful here.

In [1]:
from pathlib import Path
import pandas as pd

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / 'data/raw/energydata_complete.csv').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate data/raw/energydata_complete.csv from the current directory or its parents')

DATASET_PATH = PROJECT_ROOT / 'data' / 'raw' / 'energydata_complete.csv'
df = pd.read_csv(DATASET_PATH)
print(f'Loaded unchanged raw dataset from {DATASET_PATH.relative_to(PROJECT_ROOT)}')

Loaded unchanged raw dataset from data\raw\energydata_complete.csv


In [2]:
df.shape

(19735, 29)

In [3]:
df.head()

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


In [4]:
print('Columns:')
print(df.columns.tolist())
print('\nData types:')
df.dtypes

Columns:
['date', 'Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'rv1', 'rv2']

Data types:


date               str
Appliances       int64
lights           int64
T1             float64
RH_1           float64
T2             float64
RH_2           float64
T3             float64
RH_3           float64
T4             float64
RH_4           float64
T5             float64
RH_5           float64
T6             float64
RH_6           float64
T7             float64
RH_7           float64
T8             float64
RH_8           float64
T9             float64
RH_9           float64
T_out          float64
Press_mm_hg    float64
RH_out         float64
Windspeed      float64
Visibility     float64
Tdewpoint      float64
rv1            float64
rv2            float64
dtype: object

In [5]:
missing_by_column = df.isna().sum()
print('Missing values per column:')
display(missing_by_column)
print(f'Total missing values: {int(missing_by_column.sum())}')
print(f'Duplicate rows: {int(df.duplicated().sum())}')

Missing values per column:


date           0
Appliances     0
lights         0
T1             0
RH_1           0
T2             0
RH_2           0
T3             0
RH_3           0
T4             0
RH_4           0
T5             0
RH_5           0
T6             0
RH_6           0
T7             0
RH_7           0
T8             0
RH_8           0
T9             0
RH_9           0
T_out          0
Press_mm_hg    0
RH_out         0
Windspeed      0
Visibility     0
Tdewpoint      0
rv1            0
rv2            0
dtype: int64

Total missing values: 0
Duplicate rows: 0


In [6]:
if 'date' not in df.columns:
    raise KeyError('Required temporal column date is missing')
timestamps = pd.to_datetime(df['date'], errors='coerce')
print(f'Unparseable or missing date values: {int(timestamps.isna().sum())}')
print(f'Chronologically ordered in file: {bool(timestamps.notna().all() and timestamps.is_monotonic_increasing)}')

Unparseable or missing date values: 0
Chronologically ordered in file: True


In [7]:
duplicate_timestamp_mask = timestamps.duplicated(keep=False)
print(f'Duplicate timestamps (extra occurrences): {int(timestamps.duplicated().sum())}')
display(timestamps[duplicate_timestamp_mask].head(10))

Duplicate timestamps (extra occurrences): 0


Series([], Name: date, dtype: datetime64[us])

In [8]:
timestamp_differences = timestamps.diff().dropna()
positive_differences = timestamp_differences[timestamp_differences > pd.Timedelta(0)]
print('Most common positive timestamp differences:')
display(positive_differences.value_counts().head(10))
if not positive_differences.empty:
    actual_interval = positive_differences.mode().iloc[0]
    irregular = timestamp_differences[timestamp_differences != actual_interval]
    print(f'Most common measurement interval: {actual_interval}')
    print(f'Irregular or non-positive adjacent intervals: {len(irregular)}')
    display(irregular.head(10))
else:
    print('No positive adjacent timestamp differences are available to determine an interval.')

Most common positive timestamp differences:


date
0 days 00:10:00    19734
Name: count, dtype: int64

Most common measurement interval: 0 days 00:10:00
Irregular or non-positive adjacent intervals: 0


Series([], Name: date, dtype: timedelta64[us])

In [9]:
df.describe(include='all')

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
count,19735,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,...,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000,19735.000000
unique,19735,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,2016-01-11 17:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,97.694958,3.801875,21.686571,40.259739,20.341219,40.420420,22.267611,39.242500,20.855335,...,19.485828,41.552401,7.411665,755.522602,79.750418,4.039752,38.330834,3.760707,24.988033,24.988033
std,NaN,102.524891,7.935988,1.606066,3.979299,2.192974,4.069813,2.006111,3.254576,2.042884,...,2.014712,4.151497,5.317409,7.399441,14.901088,2.451221,11.794719,4.194648,14.496634,14.496634
min,NaN,10.000000,0.000000,16.790000,27.023333,16.100000,20.463333,17.200000,28.766667,15.100000,...,14.890000,29.166667,-5.000000,729.300000,24.000000,0.000000,1.000000,-6.600000,0.005322,0.005322
25%,NaN,50.000000,0.000000,20.760000,37.333333,18.790000,37.900000,20.790000,36.900000,19.530000,...,18.000000,38.500000,3.666667,750.933333,70.333333,2.000000,29.000000,0.900000,12.497889,12.497889
50%,NaN,60.000000,0.000000,21.600000,39.656667,20.000000,40.500000,22.100000,38.530000,20.666667,...,19.390000,40.900000,6.916667,756.100000,83.666667,3.666667,40.000000,3.433333,24.897653,24.897653
75%,NaN,100.000000,0.000000,22.600000,43.066667,21.500000,43.260000,23.290000,41.760000,22.100000,...,20.600000,44.338095,10.408333,760.933333,91.666667,5.500000,40.000000,6.566667,37.583769,37.583769


In [10]:
if 'Appliances' not in df.columns:
    raise KeyError('Required target column Appliances is missing')
target_summary = df['Appliances'].describe()
print('Appliances target summary:')
print(target_summary)
print(f"Median: {df['Appliances'].median()}")

Appliances target summary:
count    19735.000000
mean        97.694958
std        102.524891
min         10.000000
25%         50.000000
50%         60.000000
75%        100.000000
max       1080.000000
Name: Appliances, dtype: float64
Median: 60.0
